<a href="https://colab.research.google.com/github/aimeerim1/mental-health-project/blob/main/BILSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [5]:
import pandas as pd
import re

df = pd.read_csv("Combined Data.csv")
df = df.drop(columns=["Unnamed: 0"])

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.strip()
    return text

df = df.dropna()
df["label"] = df["status"].apply(lambda x: 0 if x == "Normal" else 1)
df["clean_statement"] = df["statement"].apply(clean_text)

print("Veri hazır:", df.shape)

Veri hazır: (52681, 4)


In [6]:
# Train/Test split (aynı veriyi kullanıyoruz)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_statement"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42
)

# Tokenizer
MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

print("Train shape:", X_train_pad.shape)
print("Test shape:", X_test_pad.shape)

Train shape: (42144, 100)
Test shape: (10537, 100)


In [7]:
# Model mimarisi
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 76s 124ms/step - accuracy: 0.9331 - loss: 0.1831 - val_accuracy: 0.9535 - val_loss: 0.1333
Epoch 2/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 74s 124ms/step - accuracy: 0.9649 - loss: 0.1011 - val_accuracy: 0.9516 - val_loss: 0.1357
Epoch 3/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 74s 125ms/step - accuracy: 0.9739 - loss: 0.0730 - val_accuracy: 0.9518 - val_loss: 0.1366
Epoch 4/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 75s 127ms/step - accuracy: 0.9814 - loss: 0.0528 - val_accuracy: 0.9504 - val_loss: 0.1783
Epoch 5/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 76s 127ms/step - accuracy: 0.9861 - loss: 0.0405 - val_accuracy: 0.9518 - val_loss: 0.2098


In [9]:
y_pred_prob = model.predict(X_test_pad).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

330/330 ━━━━━━━━━━━━━━━━━━━━ 11s 31ms/step
Accuracy : 0.9507449938312613
F1 Score : 0.9640805592082498
ROC-AUC  : 0.9820573556428219
